Series/IndexObj.sum() = a number
1col_DataFrame.sum() = a label + a number

In [ ]:
import pandas as pd
import numpy as np
import copy as copy
csv_file = r'D:\Python\LMHN_2025_DIRTY.csv'
# csv_file = r'D:\Python\James LMHN - 2026 - RAW.csv'
# csv_file = r'D:\Python\Kaggle\olist_geolocation_dataset.csv'
df = pd.read_csv(csv_file)
# df = df_raw.copy()



#* 0.Standardize column names
# -----------------------------------------------------------------------------
df.columns = df.columns.str.strip().str.replace(r'\s+', '_', regex=True).str.lower()
# -----------------------------------------------------------------------------


#* 1.Remove unnecessary columns
# -----------------------------------------------------------------------------
def remove_col(df, drop_col)-> pd.DataFrame:
    df = df.drop(drop_col, axis=1, errors='ignore')
    return df
# -----------------------------------------------------------------------------


#* 2.Define cols
# -----------------------------------------------------------------------------
def identify_col(df, pattern)-> str:
    get_mask = df.columns.astype('string').str.contains(pattern, case=False, regex=True, na=False)
    count_true = get_mask.sum()
    
    if count_true == 1:
        # Return the [0] cols
        col_name = df.columns[get_mask][0]
        return col_name
    elif count_true == 0:
        return None
    else:
        matched_cols = df.columns[get_mask].tolist()
        print(f"Warning: Multiple columns match '{pattern}': {matched_cols}")
        return None
# -----------------------------------------------------------------------------


# 3.Date & Time process
# -----------------------------------------------------------------------------
def chunks_maker(df: pd.DataFrame, ffilled_bfilled_date: str)-> dict:
#* Gom chunk bằng:  missing.indexs.diff().cumsum()
#* df = .zip(index, cumsum).groupby....> to_dict()
    # Indexs of NaT rows -> Index_Obj
    idx_of_nat = df.loc[df[ffilled_bfilled_date].isna(),:].index
    # Convert Index_Obj -> Series (values = Boolean after diff)
    series_error_diff = idx_of_nat.to_series().diff() > 1
    # Tổng tích lũy của Series Boolean (nếu index nối tiếp thì +0, đứt quãng +1)
    list_error_cumsum = series_error_diff.cumsum().to_list()
    # Tạo data_frame zip(index_lỗi, đánh dấu group lỗi)
    df_chunk = pd.DataFrame(zip(idx_of_nat, list_error_cumsum), columns=['idx', 'cumsum'])
    # Tạo dict groupby(cumsum)[idx] để tách các chunk ra
    df_chunk_to_dict = df_chunk.groupby('cumsum')['idx'].apply(list).to_dict()
    # Loại các chunk > 5 rows
    # filtered_chunks = {k: v if len(v) <= 3 else 'over the limit' for k, v in df_chunk_to_dict.items()}
    filtered_chunks = {k:v for k, v in df_chunk_to_dict.items() if len(v) <= 5}
    return filtered_chunks
def validate_n_correct_chunks(df: pd.DataFrame, dict_chunks: dict, raw_date: str, fill_date: str)-> list:
    pending_date_idx = []
    for _ , a_chunk in dict_chunks.items():
        p, n = min(a_chunk) - 1, max(a_chunk) + 1
        if p < 0 or n not in df.index:
            print(f"Skipping Chunk {a_chunk} = NA")
            continue
    #*------------------------
    # If prev or next is empty
        prev_pd, next_pd = df.at[p,fill_date], df.at[n,fill_date]
        if pd.isna(prev_pd) or pd.isna(next_pd):
            print(f'    prev or next is empty for chunk {a_chunk}')
            continue
    #*------------------------
    # Case 1: prev_pd == next_pd
        if prev_pd == next_pd:
            df.loc[a_chunk, fill_date] = prev_pd
            print(f'    [Match Both_day] Index {a_chunk} assigned {prev_pd.date()}')
            continue
    #*------------------------
    # Case 2.0: Stop if max gap < 4
        the_gap = (next_pd - prev_pd).days
        if the_gap >= 4:
            print(f'Chunk {a_chunk} has gap too large.')
            continue
    # Case 2.1: next_pd - prev_pd <= 3  and > 0
        if the_gap <= 3 and the_gap > 0:
    #*      Gom Set trước khi loop idx
            set_prev = {prev_pd.day, prev_pd.month, prev_pd.year}
            set_next = {next_pd.day, next_pd.month, next_pd.year}
            for idx in a_chunk:
                idx_raw = df.at[idx, raw_date]
                if pd.isna(idx_raw) or str(idx_raw).strip() == '':
                    pending_date_idx.append(idx)
                    print(f'    NGHI VẤN: Index {idx} nội dung là: {df.at[idx, raw_date]}')
                    continue
    #*      Similarity Check - (Set Validation)
                try:
                    idx_split =  idx_raw.replace(' ','/').replace('-','/').strip().split('/')
                    set_current = {int(x) for x in idx_split if x.strip().isdigit()}
                except(ValueError, TypeError, AttributeError):
                    pending_date_idx.append(idx)
                    continue

                not_prev, not_next = list(set_current - set_prev), list(set_current - set_next)

            # Nếu sau khi trừ set mà cả 2 dư ra 2 thành phần: continue
                if len(not_prev) >= 2 and len(not_next) >= 2:
                    pending_date_idx.append(idx)
                    print(f'    Something wrong at index: {idx} fill_date')
                    continue
            # If match Prev_date:
                if not not_prev:
                    df.at[idx, fill_date] = prev_pd
                    print(f'    [Match Prev_day] Index {idx} assigned {prev_pd.date()}')
            # If match Next_date:
                elif not not_next:
                    df.at[idx, fill_date] = next_pd
                    print(f'    [Match Next_day] Index {idx} assigned {next_pd.date()}')
                # Thay vì tự ghép, hãy đánh dấu để kiểm tra bằng tay
                else:
                    pending_date_idx.append(idx)
                    print(f'    NGHI VẤN: Index {idx} nội dung là: {df.at[idx, raw_date]}')
    return pending_date_idx

def time_format(df, time_col_name)-> pd.Series:      
    if time_col_name:
        t1 = pd.to_datetime(df[time_col_name], format='%I:%M%p' , errors='coerce')
        t2 = pd.to_datetime(df[time_col_name], format='%H:%M:%S', errors='coerce') 
        t3 = pd.to_datetime(df[time_col_name], format='%H:%M'   , errors='coerce') 

    return t1.fillna(t2).fillna(t3)
# -----------------------------------------------------------------------------


#* 4.Get the money cols
# -----------------------------------------------------------------------------
def get_money_col(df, pattern=r'^\s*-?\d{1,3}(?:,\d{3})*(?:\.\d+)?\s*$'):
    filtered_df = df.select_dtypes(include=['string','object'])

    def config(series):
     configuration = series.head(500).dropna().astype(str).str.strip().str.match(pattern).mean() >= 0.9
     return configuration
    
    if not filtered_df.empty:
        match = [c for c in filtered_df.columns if config(filtered_df[c])]
        return match
    else:
        return []
def comma_to_num(df):
    cols_name = get_money_col(df)
    if not cols_name:
        return []
    for col in cols_name:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')
    return cols_name
# -----------------------------------------------------------------------------  

        
# 4.Categorizing data types
# -----------------------------------------------------------------------------
def get_sample(data, before=1000, after=500)-> pd.Series:
    #! Action base on input: Series or DataFrame
    if isinstance(data, pd.Series):
        return data.head(before).dropna().head(after)
    else:
        return data.head(before).dropna(axis=0, how='all').head(after)

def is_boolean(sample: pd.Series)-> bool:
    if sample.empty: return False
    if sample.dtype == 'bool': return True
    s = sample.astype('string').str.strip().str.lower()
    x = s.isin(['true','false','yes','no','1','0']).mean()
    y = s.nunique() 
    return (x >= 0.9) and (y <= 2)

def is_datetime(sample: pd.Series)-> bool:
    if sample.empty: return False
    if sample.dtype.kind in ['b','i','u','f']: return False
    if sample.dtype.kind == 'M': return True
    x = pd.to_datetime(sample, format='mixed', errors='coerce').notna().mean()
    y = sample.str.contains(r'[-/: ]', na=False).mean()
    return (x >= 0.9) and (y >= 0.9)

def is_alo(sample: pd.Series)-> bool:
    if sample.empty: return False
    s = sample.dropna().astype('string')
    x = s.str[0].isin(['0','3','5','7','8','9']).mean()
    y = s.str.replace(r'[,\-\s]', '', regex=True).str.len().mean()
    return (x >= 0.9) and (9 <= y <= 11)

def is_money(sample: pd.Series)-> bool:
    if sample.empty: return False
    if not sample.dtype.kind == 'o': return False
    pattern = r'^\s*-?\d{1,3}(?:,\d{3})*(?:\.\d+)?\s*$'
    return sample.astype('string').str.strip().str.match(pattern, na=False).mean() >= 0.9

def is_numeric(sample: pd.Series)-> bool:
    if sample.empty: return False
    return pd.to_numeric(sample, errors='coerce').notna().mean() >= 0.9

def is_category(sample: pd.Series)-> bool:
    if sample.empty: return False
    
    numeric_ratio = pd.to_numeric(sample, errors='coerce').notna().mean()
    nunique = sample.nunique()

    # print(f"{sample.name}: numeric_ratio={numeric_ratio:.2f}, nunique={nunique}")

    if numeric_ratio > 0.8:
        return False
    return 2 <= nunique <= 33
    
def stage_0(df: pd.DataFrame)-> dict:
    #! stage_0: CAT_BY_NAME
    rules = {
    'date_col'    : r'(?:^|_)(?:date|ngay|day|created|updated)(?:_|$)',
    'time_col'    : r'(?:^|_)(?:time|gio|hour|minute|second|timestamp)(?:_|$)',
    'price'       : r'(?:^|_)(?:price|pice|unit_price|unitprice|đơn_giá|đơn_gia|gia|giá|gia_ban|giá_bán|prc)(?:_|$)',
    'numeric_col' : r'(?:^|_)(?:cost|qty|quantity|sl|disc|discount|percent|fee|rate|tax|shipping)(?:_|$)',
    'revenue'     : (r'(?:^|_)(?:revenue|total|total_amount|total_revenue|thanh_tien|thanhtien|'
                    r'doanh_thu|doanhthu|tổng_tiền|tong_tien|tongtien|grand_total|subtotal|tt|'
                    r'(?<!disc_)(?<!tax_)(?<!fee_)(?<!paid_)(?<!ship_)amount)(?:_|$)' ),
    'boolean_col' : r'(?:^|_)(?:ins_stt|tg|is_|status|active)(?:_|$)',
    'category_col': r'(?:^|_)(?:cat|type|category)(?:_|$)',
    'string_col'  : (r'(?:^|_)(?:ean|invoice|inv(?:oice|_no|_number)?|order_id|transaction_id|'
                    r'bill_no|bill_number|ma_hoa_don|serial|sku|upc|code|id)(?:_|$)' )
            }

    results = {key: [] for key in rules} | {key: [] for key in ['phone_col']}
    colname = df.columns.astype('string')
    # all_true Act like each column only has one True ticket.
    all_true = pd.Series([True] * len(colname))
    for pocket, regex in rules.items():
        # Tạo mask: str.contains() trả True/False khi tên cột CHỨA item[1] rule
        boole = colname.str.strip().str.contains(regex, case=False, na=False, regex=True)
        # Update the True from boole mask with the False from all_true
        boole = boole & all_true
        results[pocket] = colname[boole].to_list()
        # ~boole = match equal False
        all_true = all_true & ~boole
    return results, all_true, colname  
def stage_1(df: pd.DataFrame)-> dict:
    #! stage_1: CAT_BY_SAMPLE_DATA
    pocket_func = {
        'phone_col'   : [is_alo],
        'date_col'    : [is_datetime],
        'boolean_col' : [is_boolean],
        'numeric_col' : [is_money, is_numeric],
        'category_col': [is_category] 
    }
    #! unpack các biến từ stage_0
    results, used_mask, all_col = stage_0(df)

    # Chỉ xử lý các cột pending từ stage_0
    pending_cols = all_col[used_mask]
    # flag ở đây là gì chả quan trọng vì flag reset mỗi loop
    for col in pending_cols:
        # print(f'Bắt đầu {col}')
        series = get_sample(df[col])
        flag = False
        for pocket, func in pocket_func.items():
            # print(f'{col} >>> tới "{pocket}"')
            for f in func:
                # print(f'{col} thử {f.__name__}')
                if f(series):
                    results[pocket].append(col)
                    # print(f"+ Nhập '{col}' vào: {pocket} vì khớp {f.__name__} flag = True")
                    flag = True
                    break
                else: continue

            if flag:
                break
            else:
                continue
        if not flag:
            results['string_col'].append(col)
            # print(f'{col} không match => cho vào string_col')
    return results
# -----------------------------------------------------------------------------


# 5.[Revenue, Price, Qty] Validation
# -----------------------------------------------------------------------------           
def cal_rev(df: pd.DataFrame, mask: bool, payment_cols: list=None, mode: str='do_not_cal')-> pd.Series:
    #* Attribute
    # Ghi vào: obj.attrs['tên_biến'] = giá_trị
    # Lấy ra: print(obj.attrs['tên_biến'])
    # Xóa sạch: obj.attrs.clear()

    results = stage_1(df)
    qty_rules = {'qty', 'quantity', 'sl', 'so_luong'}
    list_price = results.get('price', [])

    list_qty = df.columns[[any(word in qty_rules for word in col.split('_')) or col in qty_rules for col in df.columns]]
    price_n_qty_equal_1 = len(list_price) == 1 and len(list_qty) == 1

    # Khởi tạo df_alter empty và same ở if -> elif -> return (1 flow cover all cases) Tránh lỗi UnboundLocalError.
    df_alter = pd.Series(np.nan, index=df.index)
    # mode default là bỏ qua tính df_alter
    if mode != 'do_not_cal':
        if payment_cols:
            df_alter = df.loc[mask, payment_cols].sum(axis=1)
        elif price_n_qty_equal_1:
            df_alter = df.loc[mask, list_price[0]] * df.loc[mask, list_qty[0]]
    # Ghi kèm hướng dẫn sd, phải để cuối cùng chứ cho lên .loc là mất hết
    df_alter.attrs['results'] = results
    df_alter.attrs['price_n_qty_equal_1'] = price_n_qty_equal_1
    return df_alter
def rev_validate(df, payment_cols=None)-> pd.DataFrame:

    mask = pd.Series(True, index=df.index)
    cal_results = cal_rev(df, mask, payment_cols)
    results = copy.deepcopy(cal_results.attrs.get('results', {}))
    price_n_qty_equal_1 = cal_results.attrs.get('price_n_qty_equal_1', False)

    key = 'revenue'
    list_rev = results.get(key, [])
    # rev = list_rev[0] Nguy hiểm vì nếu list_rev rông thì lỗi

    if len(list_rev) == 0:
        rev = key
        if rev not in df.columns:
            df[rev] = cal_rev(df, mask, payment_cols, mode='cal')
            print(f"Created new column: {rev}")

    elif len(list_rev) == 1:
        rev = list_rev[0]
        rev_val = pd.to_numeric(df[rev], errors='coerce').notna().mean()
        rev_na = df[rev].isna().sum()
        print(f'Detected: 1 revenue column ({rev_val*100:.2f}% valid)')
        if rev_val < 0.99:
            if payment_cols or price_n_qty_equal_1:
                mask = df[rev].isna()
                df.loc[mask, rev] = cal_rev(df, mask, payment_cols, mode='cal')
                print(f'Revenue gaps: {rev_na} filled')
            else:
                print(f'Insufficient data to fill in {rev_na} revenue gaps.')
    
        rev_mean = df[rev].mean()
        mask = pd.Series(True, index=df.index)
        if payment_cols or price_n_qty_equal_1:
            rev_alter_mean = cal_rev(df, mask, payment_cols, mode='cal').mean()
            if rev_alter_mean != 0:
                print(f'revenue(mean) / alter_revenue(mean) = {(rev_mean/rev_alter_mean)*100:.2f}%')
    else:
        print(f"Cảnh báo: Có nhiều cột {key}: {list_rev}")
    return df, results
# -----------------------------------------------------------------------------   


#* 6. Date Validation & recover
# -----------------------------------------------------------------------------  
def recover_date(df: pd.DataFrame, date_raw: str, anchor_col_name: str=None)-> list:
    if not df.index.is_monotonic_increasing:
        df = df.sort_index()
        print("Warning: Index was not sorted. DataFrame has been sorted automatically.")

    list_if_error = []

    if date_raw:
        d1 = pd.to_datetime(df[date_raw], format='%Y-%m-%d', errors='coerce')
        d2 = pd.to_datetime(df[date_raw], format='%d-%m-%Y', errors='coerce')
        df['fill_date'] = d1.fillna(d2)

#* Nếu có Anchor_col > ffill, bfill trước.
    if anchor_col_name:
        df['fill_date'] = df.groupby(anchor_col_name)['fill_date'].transform('ffill')
        df['fill_date'] = df.groupby(anchor_col_name)['fill_date'].transform('bfill')
        print(f'Missing date detected, proceed ffill & bffll by anchor "{anchor_col_name}"')

    number_of_errors_date = df['fill_date'].isna().sum()
    if number_of_errors_date > 0:   
        the_chunks = chunks_maker(df, 'fill_date')
        if the_chunks:
            list_if_error = validate_n_correct_chunks(df, the_chunks, date_raw, 'fill_date')
    if not list_if_error:
        print(f"Number of NaT in 'fill_date': {df['fill_date'].isna().sum()}")
    return list_if_error


def execution(df: pd.DataFrame, final_results: dict)-> pd.DataFrame:
    df_new = pd.DataFrame()
    results = final_results
    
    for key, cols in results.items():
        if not cols:
            continue
        if key == 'date_col':
            for c in cols:
                df_new[c] = recover_date(df, c)
        if key == 'time_col':
            for c in cols:
                df_new[c] = time_format(df, c)
        if key == 'price':
            df_new[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
        if key == 'revenue':
            df_new[cols] = df[cols]
        if key == 'numeric_col':
            df_new[cols] = df[cols].apply(pd.to_numeric, errors='coerce').fillna(0)
        if key == 'boolean_col':
            df_new[cols] = df[cols].astype('boolean')
        if key == 'category_col':
            df_new[cols] = df[cols].astype('category')
        if key == 'string_col':
            df_new[cols] = df[cols].astype('string').fillna('unknown')
        if key == 'phone_col':
            df_new[cols] = df[cols].astype('string').fillna('unknown')
    try:
        df_new = df_new[df.columns]
    except KeyError as e:
        print(f"🫠 Execution > Mất cột: {e}")
    return df_new

payment_cols = ['cash', 'card', 'payoo', 'banking', 'mkt', 'vnpay', 'trade_in']
df, res = rev_validate(df, payment_cols)
df_new = execution(df, res)













# clean_date = recover_date(df, 'date', 'invoice')
df_new.info()
# df_new

# Dọn date_temp , sát nhập fill_date

Detected: 1 revenue column (85.14% valid)
Revenue gaps: 1800 filled
revenue(mean) / alter_revenue(mean) = 100.43%
    [Match Prev_day] Index 253 assigned 2025-01-10
    [Match Both_day] Index [371, 372, 373] assigned 2025-01-14
    [Match Both_day] Index [7437] assigned 2025-09-21
    [Match Both_day] Index [7464, 7465, 7466] assigned 2025-09-22
    [Match Both_day] Index [7496, 7497, 7498, 7499] assigned 2025-09-23
    [Match Both_day] Index [8906] assigned 2025-10-20
Number of NaT in 'fill_date': 0
🫠 Execution > Mất cột: "['fill_date'] not in index"
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12113 entries, 0 to 12112
Data columns (total 28 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date             0 non-null      float64       
 1   time             10290 non-null  datetime64[ns]
 2   price            11925 non-null  float64       
 3   qty              12113 non-null  float64       
 4   ins_fee   

In [6]:
df.at[371,'date']
# df_test = pd.DataFrame()
# df_test['Hello'] = pd.Series([True]*20)
# df_new = execution(df)
# df_new.median(numeric_only=True)
# df_new.mean(numeric_only=True).round()
# df_new['total_payment'] = df_new[['cash', 'card', 'payoo', 'banking', 'mkt', 'vnpay', 'trade_in']].sum(axis=1)
# rev = df_new['revenue']
# tol =  df_new['total_payment']
# # df.loc[mask, cols_list]: Mask hoạt động như label index
# # df_new.loc[df_new['revenue'].isna(), ['invoice', 'sa', 'revenue', 'cash', 'card', 'payoo', 'banking', 'mkt', 'vnpay', 'total_payment']]

'25-01-14'

In [ ]:
# list_idx = date_errors.to_list()
# now_stage = 0
# pre_idx = 0
# resu = []
# for idx in list_idx:
#     if resu:
#         if idx - pre_idx >= 2:
#             now_stage += 1
#     resu.append(now_stage)
#     pre_idx = idx      

In [ ]:
# def auto_fill(df):
#     dict_name = stage_1(df)
#     # print(f"{{col: 0 for col in dict_name['{d}']}} |")
#     fill_rules = (
#         {col: pd.NaT for col in dict_name['date_col']} |
#         {col: pd.NaT for col in dict_name['time_col']} |
#         {col: 0 for col in dict_name['price']} |
#         {col: 0 for col in dict_name['revenue']} |
#         {col: 0 for col in dict_name['numeric_col']} |
#         {col: 0 for col in dict_name['boolean_col']} |
#         {col: "N/A" for col in dict_name['category_col']} |
#         {col: "N/A" for col in dict_name['string_col']} |
#         {col: "unknown" for col in dict_name['phone_col']} )
#     return df